In [2]:
import io
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# 1. Download official Cricsheet T20 International archive
cricsheet_url = "https://cricsheet.org/downloads/t20s_male_csv2.zip"
print("Downloading official T20 International archive from Cricsheet...")

req = urllib.request.Request(cricsheet_url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as response:
    zip_bytes = io.BytesIO(response.read())

# 2. Scan the archive and extract ONLY matches involving Sri Lanka
print("Scanning archive and extracting Sri Lanka matches...")
sl_match_frames = []

with zipfile.ZipFile(zip_bytes) as z:
    # Get all match ball-by-ball files (excluding info files and txt files)
    match_files = [f for f in z.namelist() if f.endswith('.csv') and not f.endswith('_info.csv')]

    for filename in match_files:
        raw_content = z.read(filename)
        # Fast byte check: only parse if Sri Lanka is mentioned in the match file
        if b'Sri Lanka' in raw_content:
            df_match = pd.read_csv(io.BytesIO(raw_content), low_memory=False)
            if ('Sri Lanka' in df_match['batting_team'].values) or ('Sri Lanka' in df_match['bowling_team'].values):
                sl_match_frames.append(df_match)

# 3. Combine all Sri Lanka matches into one master dataframe
sl_matches = pd.concat(sl_match_frames, ignore_index=True)

print("\n=== ✅ Sri Lanka T20 International Data Loaded Successfully! ===")
print(f"Total T20 Matches Extracted: {sl_matches['match_id'].nunique()}")
print(f"Total Deliveries (Balls) in Dataset: {len(sl_matches):,}")
print(f"Date Range: {sl_matches['start_date'].min()} to {sl_matches['start_date'].max()}\n")

# Preview first 5 balls
sl_matches[['match_id', 'start_date', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'bowler', 'runs_off_bat', 'extras']].head()

Scanning archive and extracting Sri Lanka matches...

=== ✅ Sri Lanka T20 International Data Loaded Successfully! ===
Total T20 Matches Extracted: 225
Total Deliveries (Balls) in Dataset: 52,494
Date Range: 2006-06-15 to 2026-09-17



,match_id,start_date,innings,ball,batting_team,bowling_team,striker,bowler,runs_off_bat,extras
0,1001349,2017-02-17,1,0.1,Australia,Sri Lanka,AJ Finch,SL Malinga,0,0
1,1001349,2017-02-17,1,0.2,Australia,Sri Lanka,AJ Finch,SL Malinga,0,0
2,1001349,2017-02-17,1,0.3,Australia,Sri Lanka,AJ Finch,SL Malinga,1,0
3,1001349,2017-02-17,1,0.4,Australia,Sri Lanka,M Klinger,SL Malinga,2,0
4,1001349,2017-02-17,1,0.5,Australia,Sri Lanka,M Klinger,SL Malinga,0,0


In [4]:
# 1. Define match phases based on the over number
# In Cricsheet, ball numbers are recorded as decimals: 0.1 = Over 1, 15.1 = Over 16
def get_phase(ball):
    over = int(ball)
    if over < 6:
        return 'Powerplay'
    elif over < 15:
        return 'Middle'
    else:
        return 'Death'

sl_matches['phase'] = sl_matches['ball'].apply(get_phase)

# 2. Filter for deliveries where Sri Lanka was BOWLING in the DEATH phase (overs 16-20)
sl_death = sl_matches[(sl_matches['bowling_team'] == 'Sri Lanka') & (sl_matches['phase'] == 'Death')].copy()

# 3. Clean cricket rules for bowler statistics:
# Bowler conceded runs = runs off bat + wides + noballs (byes and legbyes don't count against bowler)
sl_death['wides'] = sl_death['wides'].fillna(0)
sl_death['noballs'] = sl_death['noballs'].fillna(0)
sl_death['bowler_runs'] = sl_death['runs_off_bat'] + sl_death['wides'] + sl_death['noballs']

# Legal deliveries: excludes wides and noballs
sl_death['is_legal'] = (sl_death['wides'] == 0) & (sl_death['noballs'] == 0)

# Bowler wickets: excludes runouts, retired hurt, etc.
bowler_wickets = ['bowled', 'caught', 'caught and bowled', 'lbw', 'stumped', 'hit wicket']
sl_death['is_wicket'] = sl_death['wicket_type'].isin(bowler_wickets).astype(int)

# Dot balls: zero runs conceded
sl_death['is_dot'] = (sl_death['runs_off_bat'] == 0) & (sl_death['extras'] == 0)

# 4. Group by Bowler to calculate quantitative metrics
death_bowlers = sl_death.groupby('bowler').agg(
    total_balls=('is_legal', 'sum'),
    runs_conceded=('bowler_runs', 'sum'),
    wickets=('is_wicket', 'sum'),
    dot_balls=('is_dot', 'sum')
).reset_index()

# 5. Filter for bowlers with at least 60 balls (10 overs) in the death phase
death_bowlers = death_bowlers[death_bowlers['total_balls'] >= 60].copy()

# Calculate advanced metrics
death_bowlers['economy_rate'] = round((death_bowlers['runs_conceded'] / death_bowlers['total_balls']) * 6, 2)
death_bowlers['strike_rate'] = round(death_bowlers['total_balls'] / death_bowlers['wickets'], 2)
death_bowlers['dot_ball_pct'] = round((death_bowlers['dot_balls'] / death_bowlers['total_balls']) * 100, 1)

# 6. Interactive Plotly Quadrant Chart
fig = px.scatter(
    death_bowlers,
    x='economy_rate',
    y='strike_rate',
    size='wickets',
    color='dot_ball_pct',
    text='bowler',
    hover_name='bowler',
    hover_data={
        'total_balls': True,
        'wickets': True,
        'economy_rate': ':.2f',
        'strike_rate': ':.2f',
        'dot_ball_pct': ':.1f%'
    },
    title='<b>Sri Lanka T20I Death Bowling Matrix (Overs 16–20)</b><br><sup>Minimum 60 balls bowled | Bottom-Left Quadrant = Elite (Low Economy + Fast Wickets)</sup>',
    labels={
        'economy_rate': 'Death Economy Rate (Runs per Over) → Lower is Better',
        'strike_rate': 'Balls per Wicket (Strike Rate) → Lower is Better',
        'dot_ball_pct': 'Dot Ball %'
    },
    color_continuous_scale='Viridis',
    template='plotly_white',
    height=650
)

# Position names cleanly next to bubbles
fig.update_traces(textposition='top center', textfont=dict(size=10, color='#2C3E50'))

# Add reference lines for global averages
fig.add_vline(x=9.5, line_dash="dash", line_color="#E74C3C", annotation_text="Global Avg Economy (~9.5)")
fig.add_hline(y=14.0, line_dash="dash", line_color="#E74C3C", annotation_text="Global Avg Strike Rate (~14 balls)")

fig.show()

In [5]:
# 1. Filter for deliveries where Sri Lanka was BATTING
sl_batting = sl_matches[sl_matches['batting_team'] == 'Sri Lanka'].copy()

# 2. Exclude wides for balls faced calculation (wides do not count as a ball faced by batsman)
sl_batting['is_ball_faced'] = sl_batting['wides'].isna() | (sl_batting['wides'] == 0)

# 3. Calculate Runs and Balls Faced per Batsman per Phase
batsman_phase = sl_batting.groupby(['striker', 'phase']).agg(
    runs=('runs_off_bat', 'sum'),
    balls=('is_ball_faced', 'sum')
).reset_index()

# 4. Calculate Strike Rate = (Runs / Balls) * 100
batsman_phase['strike_rate'] = round((batsman_phase['runs'] / batsman_phase['balls']) * 100, 1)

# 5. Pivot table to compare Powerplay, Middle, and Death side-by-side
phase_pivot = batsman_phase.pivot(index='striker', columns='phase', values=['runs', 'balls', 'strike_rate'])

# Flatten column names
phase_pivot.columns = [f"{col[1]}_{col[0]}" for col in phase_pivot.columns]
phase_pivot = phase_pivot.reset_index()

# Total balls faced across all phases
total_balls = sl_batting.groupby('striker')['is_ball_faced'].sum().reset_index(name='total_career_balls')
phase_pivot = phase_pivot.merge(total_balls, on='striker')

# 6. Filter for established batsmen (minimum 150 career balls faced)
established_batsmen = phase_pivot[phase_pivot['total_career_balls'] >= 150].copy()

# Clean missing values for batsmen who rarely bat in specific phases
established_batsmen = established_batsmen.fillna(0)

# Calculate "Death Acceleration" = (Death Strike Rate - Powerplay Strike Rate)
established_batsmen['death_acceleration'] = established_batsmen['Death_strike_rate'] - established_batsmen['Powerplay_strike_rate']

# 7. Interactive Scatter Plot: Powerplay SR vs Death SR
fig_bat = px.scatter(
    established_batsmen[established_batsmen['Death_balls'] >= 30], # At least 30 balls faced at death
    x='Powerplay_strike_rate',
    y='Death_strike_rate',
    size='total_career_balls',
    color='death_acceleration',
    text='striker',
    hover_name='striker',
    hover_data={
        'Powerplay_strike_rate': ':.1f',
        'Middle_strike_rate': ':.1f',
        'Death_strike_rate': ':.1f',
        'death_acceleration': ':.1f',
        'total_career_balls': True
    },
    title='<b>Sri Lanka T20I Batting: Powerplay Intent vs. Death Acceleration</b><br><sup>Min 150 career balls | Top-Right = High Impact in Both Phases | Color = Acceleration from PP to Death</sup>',
    labels={
        'Powerplay_strike_rate': 'Powerplay Strike Rate (Overs 1-6)',
        'Death_strike_rate': 'Death Overs Strike Rate (Overs 16-20)',
        'death_acceleration': 'SR Boost at Death'
    },
    color_continuous_scale='Spectral',
    template='plotly_white',
    height=650
)

fig_bat.update_traces(textposition='top center', textfont=dict(size=10, color='#2C3E50'))

# Add reference lines for standard benchmark strike rates
fig_bat.add_vline(x=120.0, line_dash="dash", line_color="#7F8C8D", annotation_text="Benchmark PP SR (120)")
fig_bat.add_hline(y=150.0, line_dash="dash", line_color="#7F8C8D", annotation_text="Benchmark Death SR (150)")

fig_bat.show()

In [6]:
# 1. Calculate Batting Impact for all Sri Lankan batsmen (min 100 balls)
sl_batting_all = sl_matches[sl_matches['batting_team'] == 'Sri Lanka'].copy()
sl_batting_all['is_ball'] = sl_batting_all['wides'].isna() | (sl_batting_all['wides'] == 0)

bat_stats = sl_batting_all.groupby('striker').agg(
    total_runs=('runs_off_bat', 'sum'),
    balls_faced=('is_ball', 'sum')
).reset_index()

bat_stats = bat_stats[bat_stats['balls_faced'] >= 100].copy()
bat_stats['bat_sr'] = (bat_stats['total_runs'] / bat_stats['balls_faced']) * 100

# Batting Impact Formula: (Runs * SR) / 100
bat_stats['bat_impact'] = round((bat_stats['total_runs'] * (bat_stats['bat_sr'] / 100)), 1)

# 2. Calculate Bowling Impact for all Sri Lankan bowlers (min 120 balls / 20 overs)
sl_bowling_all = sl_matches[sl_matches['bowling_team'] == 'Sri Lanka'].copy()
sl_bowling_all['wides'] = sl_bowling_all['wides'].fillna(0)
sl_bowling_all['noballs'] = sl_bowling_all['noballs'].fillna(0)
sl_bowling_all['bowler_runs'] = sl_bowling_all['runs_off_bat'] + sl_bowling_all['wides'] + sl_bowling_all['noballs']
sl_bowling_all['is_legal'] = (sl_bowling_all['wides'] == 0) & (sl_bowling_all['noballs'] == 0)
bowler_wickets = ['bowled', 'caught', 'caught and bowled', 'lbw', 'stumped', 'hit wicket']
sl_bowling_all['is_wicket'] = sl_bowling_all['wicket_type'].isin(bowler_wickets).astype(int)

bowl_stats = sl_bowling_all.groupby('bowler').agg(
    balls_bowled=('is_legal', 'sum'),
    runs_conceded=('bowler_runs', 'sum'),
    wickets=('is_wicket', 'sum')
).reset_index()

bowl_stats = bowl_stats[bowl_stats['balls_bowled'] >= 120].copy()
bowl_stats['economy'] = (bowl_stats['runs_conceded'] / bowl_stats['balls_bowled']) * 6

# Bowling Impact Formula: (Wickets * 25) + (Benchmark Econ 8.0 - Bowler Econ) * (Balls / 6)
bowl_stats['bowl_impact'] = round((bowl_stats['wickets'] * 25) + (8.0 - bowl_stats['economy']) * (bowl_stats['balls_bowled'] / 6), 1)

# Preview top 5 batsmen and bowlers by impact
print("=== Top 5 Batsmen by Quantitative Impact ===")
print(bat_stats.sort_values(by='bat_impact', ascending=False)[['striker', 'total_runs', 'bat_sr', 'bat_impact']].head())

print("\n=== Top 5 Bowlers by Quantitative Impact ===")
print(bowl_stats.sort_values(by='bowl_impact', ascending=False)[['bowler', 'wickets', 'economy', 'bowl_impact']].head())

=== Top 5 Batsmen by Quantitative Impact ===
        striker  total_runs      bat_sr  bat_impact
68   P Nissanka        2606  127.807749      3330.7
6    BKG Mendis        2510  129.782834      3257.5
57  MDKJ Perera        2307  132.815199      3064.0
55   MD Shanaka        2008  129.381443      2598.0
95   TM Dilshan        1747  120.068729      2097.6

=== Top 5 Bowlers by Quantitative Impact ===
             bowler  wickets   economy  bowl_impact
62     PWH de Silva      155  7.256286       4131.3
69       SL Malinga      108  7.430464       2872.0
61     PVD Chameera       98  8.160827       2403.3
43     M Theekshana       83  7.074335       2359.3
33  KMDN Kulasekara       64  7.425190       1713.3


In [7]:
from scipy.optimize import milp, LinearConstraint, Bounds

# 1. Merge Batting and Bowling stats into a single Master Player Pool
player_pool = pd.merge(
    bat_stats[['striker', 'total_runs', 'bat_sr', 'bat_impact']].rename(columns={'striker': 'player'}),
    bowl_stats[['bowler', 'wickets', 'economy', 'bowl_impact']].rename(columns={'bowler': 'player'}),
    on='player',
    how='outer'
).fillna(0)

# Calculate Total Composite Impact (Batting Impact + Bowling Impact)
player_pool['total_impact'] = player_pool['bat_impact'] + player_pool['bowl_impact']

# Filter out fringe players with very low combined contribution (min impact 300)
player_pool = player_pool[player_pool['total_impact'] >= 300].reset_index(drop=True)

# 2. Tag Player Capabilities based on cricket roles
wicketkeepers = ['KC Sangakkara', 'BKG Mendis', 'MDKJ Perera', 'LD Chandimal', 'N Dickwella', 'S Samarawickrama']
spinners = ['PWH de Silva', 'M Theekshana', 'BAW Mendis', 'SMSM Senanayake', 'A Dananjaya', 'DN Wellalage']
pacers = ['SL Malinga', 'PVD Chameera', 'KMDN Kulasekara', 'M Pathirana', 'CBRSL Kumara', 'D Madushanka', 'NLTC Perera', 'I Udana', 'B Fernando', 'N Pradeep', 'CAK Rajitha']

player_pool['is_wk'] = player_pool['player'].isin(wicketkeepers).astype(int)
player_pool['is_spinner'] = player_pool['player'].isin(spinners).astype(int)
player_pool['is_pacer'] = player_pool['player'].isin(pacers).astype(int)
# Capable bowlers: any player who bowled at least 20 career overs
player_pool['is_bowler'] = (player_pool['wickets'] >= 10).astype(int)

# 3. Formulate the MILP Optimization Problem
n_players = len(player_pool)

# Objective: Minimize -Total_Impact (which maximizes Total Impact)
c = -player_pool['total_impact'].values

# Constraints Matrix:
# Row 0: Exactly 11 players
# Row 1: At least 1 Wicketkeeper
# Row 2: At least 2 Pacers
# Row 3: At least 1 Spinner
# Row 4: At least 5 Bowlers (minimum 20 overs capacity)
A = np.array([
    np.ones(n_players),
    player_pool['is_wk'].values,
    player_pool['is_pacer'].values,
    player_pool['is_spinner'].values,
    player_pool['is_bowler'].values
])

# Bounds for the constraints [lower_bound, upper_bound]
lhs = [11, 1, 2, 1, 5]
rhs = [11, np.inf, np.inf, np.inf, np.inf]
constraints = LinearConstraint(A, lhs, rhs)

# Decision variables must be binary integers: x_i in {0, 1}
integrality = np.ones(n_players) # 1 = integer variable
bounds = Bounds(0, 1)

# 4. Solve the Optimization Problem!
res = milp(c=c, constraints=constraints, integrality=integrality, bounds=bounds)

# 5. Extract the Selected Dream XI
selected_indices = np.where(res.x > 0.5)[0]
dream_xi = player_pool.iloc[selected_indices].copy()

# Sort the team: Pure batsmen first, all-rounders in middle, specialist bowlers at the bottom
dream_xi = dream_xi.sort_values(by=['is_bowler', 'total_runs'], ascending=[True, False]).reset_index(drop=True)

print("==========================================================================")
print("     🏏 THE MATHEMATICALLY OPTIMAL ALL-TIME SRI LANKA T20 XI 🇱🇰")
print("          (Solved via Binary Integer Linear Programming)")
print("==========================================================================")
for i, row in dream_xi.iterrows():
    role_desc = []
    if row['is_wk']: role_desc.append("WK")
    if row['is_pacer']: role_desc.append("Pace")
    if row['is_spinner']: role_desc.append("Spin")
    if row['bat_impact'] > 1500 and row['bowl_impact'] > 1000: role_desc.append("All-Rounder")

    role_str = f"({', '.join(role_desc)})" if role_desc else "(Batter)"
    print(f"{i+1:2d}. {row['player']:<18} | Runs: {int(row['total_runs']):<5} | Wkts: {int(row['wickets']):<3} | Total Impact: {row['total_impact']:>7.1f} {role_str}")

print("--------------------------------------------------------------------------")
print(f"Total Team Impact Score : {-res.fun:,.1f} points")
print(f"Optimization Status     : {res.message}")
print("==========================================================================")

     🏏 THE MATHEMATICALLY OPTIMAL ALL-TIME SRI LANKA T20 XI 🇱🇰
          (Solved via Binary Integer Linear Programming)
 1. P Nissanka         | Runs: 2606  | Wkts: 0   | Total Impact:  3330.7 (Batter)
 2. BKG Mendis         | Runs: 2510  | Wkts: 0   | Total Impact:  3257.5 (WK)
 3. MDKJ Perera        | Runs: 2307  | Wkts: 0   | Total Impact:  3064.0 (WK)
 4. TM Dilshan         | Runs: 1747  | Wkts: 9   | Total Impact:  2353.6 (Batter)
 5. MD Shanaka         | Runs: 2008  | Wkts: 42  | Total Impact:  3602.7 (All-Rounder)
 6. AD Mathews         | Runs: 1343  | Wkts: 41  | Total Impact:  2779.0 (All-Rounder)
 7. NLTC Perera        | Runs: 1042  | Wkts: 42  | Total Impact:  2364.4 (Pace)
 8. PWH de Silva       | Runs: 742   | Wkts: 155 | Total Impact:  5022.2 (Spin)
 9. SL Malinga         | Runs: 136   | Wkts: 108 | Total Impact:  2986.9 (Pace)
10. PVD Chameera       | Runs: 125   | Wkts: 98  | Total Impact:  2495.2 (Pace)
11. M Theekshana       | Runs: 114   | Wkts: 83  | Total Impact:  